In [1]:
from Bio import PDB
from scipy.spatial import KDTree

def read_pdb_coordinates(pdb_file):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('structure', pdb_file)
    
    atoms = []
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    atoms.append(atom)
    
    return atoms

def find_nearest_neighbors(oxygens, phosphates, distance_threshold=4.0):
    # Create KDTree for oxygens
    oxygen_coords = [atom.coord for atom in oxygens]
    oxygen_tree = KDTree(oxygen_coords)

    # Create KDTree for phosphates
    phosphate_coords = [atom.coord for atom in phosphates]
    phosphate_tree = KDTree(phosphate_coords)

    # Find nearest neighbors
    oxygen_phosphate_pairs = []
    for oxygen_idx in range(len(oxygen_coords)):
        nearest_phosphate_dist, nearest_phosphate_idx = phosphate_tree.query(oxygen_coords[oxygen_idx], k=1)
        nearest_oxygen_dist, nearest_oxygen_idx = oxygen_tree.query(phosphate_coords[nearest_phosphate_idx], k=1)
        
        # Check if the oxygen and phosphate are within the distance threshold
        if nearest_phosphate_dist <= distance_threshold and nearest_oxygen_idx == oxygen_idx:
            oxygen_atom = oxygens[oxygen_idx]
            phosphate_atom = phosphates[nearest_phosphate_idx]
            oxygen_phosphate_pairs.append((oxygen_atom, phosphate_atom))
    
    return oxygen_phosphate_pairs

def write_pdb_format(output_file, pairs):
    with open(output_file, 'w') as f:
        for oxygen, phosphate in pairs:
            f.write(f"ATOM  {oxygen.get_id():>5} {oxygen.get_name():<4} {oxygen.get_parent().resname:<3} {oxygen.get_parent().get_id()}  {oxygen.coord[0]:>8.3f}{oxygen.coord[1]:>8.3f}{oxygen.coord[2]:>8.3f}  1.00 22.78      B    C\n")
            f.write(f"ATOM  {phosphate.get_id():>5} {phosphate.get_name():<4} {phosphate.get_parent().resname:<3} {phosphate.get_parent().get_id()}  {phosphate.coord[0]:>8.3f}{phosphate.coord[1]:>8.3f}{phosphate.coord[2]:>8.3f}  1.00 22.78      B    O\n")
        f.write("TER\n")

if __name__ == "__main__":
    oxygen_files = ['/home/tracyy2/HA_binder_design/HAp_010_row2_OE1.pdb', '/home/tracyy2/HA_binder_design/HAp_010_row2_OE2.pdb']
    phosphate_file = '/home/tracyy2/HA_binder_design/HAp_010_row2_CD.pdb'
    
    # Read coordinates from PDB files
    oxygens = []
    for file in oxygen_files:
        oxygens.extend(read_pdb_coordinates(file))
    phosphates = read_pdb_coordinates(phosphate_file)
    
    # Find pairs of oxygens and phosphates within 3.0 Angstroms
    pairs = find_nearest_neighbors(oxygens, phosphates, distance_threshold=3.0)
    
    # Write results to output file in PDB format
    output_file = "output.pdb"
    write_pdb_format(output_file, pairs)
    
    print(f"Output written to {output_file}")


Output written to output.pdb
